---
title: xarray for the Open Data Cube
short_title: xarray for ODC
subject: Beginner Guide
subtitle: Working with the xarray.Dataset returned by dc.load.
description: Working with the xarray.Dataset returned by dc.load.
keywords:
  - open-data-cube
  - odc
  - xarray
  - beginner-guide
---

This notebook introduces `xarray.Dataset` and `xarray.DataArray`, the objects that hold data returned by `dc.load`.
It explains how their labels describe the data, then uses those labels to select subsets and calculate a new variable.[^edits]

[^edits]: Tutorial notebooks update automatically; edits to a tutorial notebook may be overwritten on the next update.
    Keep a working copy in a separate file to preserve changes.

## A. Objectives

- Load a sample `Dataset`
- Interpret its dimensions, coordinates, and data variables
- Select subsets by label and by index
- Calculate a new `DataArray` with arithmetic

## B. The xarray Dataset and DataArray

A satellite observation records a grid of pixel values for one band at one point in time.
An Earth observation query commonly requests several bands and dates for the same area, producing a set of grids that must remain correctly aligned.

A plain numerical array stores the values but does not identify what each axis or position represents.
Dates, pixel positions, band names, and projection information would have to be tracked separately.
xarray keeps this descriptive information with the values, which makes the data easier to inspect and safer to combine.

Each named axis is a **dimension**.
The data loaded in this notebook uses `time`, `y`, and `x`: time identifies the observation date, while y and x locate pixels on the spatial grid.
A **coordinate** supplies the labels along a dimension, such as dates along `time` or projected positions along `y` and `x`.

xarray organises these labelled values in two main objects:

- A **DataArray** holds one named variable together with its dimensions and coordinates.
  In this notebook, a band such as `red` is a DataArray with dimensions `time`, `y`, and `x`.
  Its values therefore remain associated with the correct dates and pixel positions during selection and arithmetic.

- A **Dataset** groups related DataArrays.
  Each requested band (`red`, `green`, `blue`, and `nir`) is a data variable in the Dataset returned by `dc.load`.
  These variables use the same `time`, `y`, and `x` grid, so values at matching coordinates refer to the same date and location.
  A Dataset can also carry attributes that describe the collection as a whole, including its map projection.

![One DataArray drawn as a single labelled grid over y/latitude and x/longitude, alongside one Dataset drawn as a matrix of four bands (red, green, blue, nir) across four dates, with all bands sharing the same y, x grid.](../../assets/xarray-dataarray-vs-dataset.svg)

`dc.load` returns an xarray Dataset.
The next sections load a small example and relate each part of its display to this structure.
The xarray documentation provides the formal definitions: [xarray data structures](https://docs.xarray.dev/en/stable/user-guide/data-structures.html).

## C. Loading a sample Dataset

The query below loads annual GeoMAD data for four bands over a small area.
The result is assigned to `ds`, a conventional abbreviation for dataset.

In [ ]:
from datacube import Datacube

dc = Datacube(app="xarray_for_odc")

query = {
    "product": "s2_geomad_annual",
    "x": (98.80, 98.90),
    "y": (2.65, 2.55),
    "time": ("2024", "2025"),
    "measurements": ["red", "green", "blue", "nir"],
    "output_crs": "EPSG:32647",
    "resolution": (-30, 30),
}

ds = dc.load(**query)
ds

## D. Reading the Dataset structure

The displayed Dataset separates its description into dimensions, data variables, coordinates, and attributes.
Reading these parts first confirms what the query loaded before any analysis begins.

![Structure of an xarray.Dataset returned by dc.load, showing the four parts: dimensions, data variables, coordinates, and attributes.](../../assets/xarray-dataset-structure.svg)

1. **Dimensions and sizes** describe the axes of the data and their lengths.

In [ ]:
ds.sizes

The names `time`, `y`, and `x` are the dimensions.
For this result, their sizes show two time slices on a 369 × 372 pixel grid.
Together with the requested 30 m resolution, these values give an immediate sense of the area represented and the amount of data involved.

2. **Data variables** are the measured values held in the Dataset.

In [ ]:
ds.data_vars

The four data variables are the red, green, blue, and near-infrared surface-reflectance bands requested in the query.
For each variable, this view reports its dimensions, shape, data type, estimated size, and a short preview of its values.
Those details help reveal how a measurement is stored, such as whether surface temperature contains raw digital numbers or values already converted to degrees.

Selecting one variable from `data_vars` returns its DataArray and exposes its own dimensions, coordinates, values, and attributes:

In [ ]:
ds.data_vars["red"]

3. **Coordinates** provide the labels used to locate data along each dimension.

In [ ]:
ds.coords

The `time` coordinate contains the dates, while `y` and `x` contain projected pixel positions.
These labels support selection and allow xarray to align values from the same dates and locations during a calculation.

4. **Attributes** hold descriptive metadata that applies to the Dataset as a whole.

In [ ]:
ds.attrs

The requested CRS is EPSG:32647 (WGS 84 / UTM zone 47N).
This projected CRS uses metres, so differences between values on the `y` and `x` coordinates can be read as distances without first reprojecting the data.

## E. Selecting subsets of the Dataset

Selection extracts the part of a Dataset or DataArray needed for the next calculation and leaves the original `ds` unchanged.
The choice between `.sel` and `.isel` depends on whether the required position is known by its coordinate label or by its integer index.

Both are methods, which are functions attached to an object and called with dot notation.
For example, `ds.sel(time="2024")` uses a time label, whereas `ds.isel(time=0)` uses a position.
Inside each call, `time=value` identifies the dimension and the label or index to retain.

### 1. Selection by label

`.sel` looks up values by their coordinate labels.
xarray understands date-like labels, so `time` can be selected with a year, a complete date, a list of dates, or a date range.

In [ ]:
ds.sel(time="2024")

The year label selects the time coordinate that falls in 2024.
The result is a new Dataset containing that time slice, with the y and x grids unchanged.

Other label-based selections use the same method:

- a list of labels: `ds.sel(time=["2024-01-01", "2025-01-01"])`
- a range slice: `ds.sel(time=slice("2024", "2025"))`
- a boolean mask: `ds.sel(time=ds.time.dt.year == 2024)`

### 2. Selection by index

`.isel` looks up values by integer position, following the same zero-based convention as NumPy.

In [ ]:
ds.isel(time=0)

Index `0` selects the first position along the `time` dimension, regardless of its date label.
A list of indices (`ds.isel(time=[0, 1])`) or a slice (`ds.isel(time=slice(0, 2))`) selects several positions.
This approach is useful when the required position is known but its coordinate label is not.

## F. Arithmetic across variables

Each data variable in the Dataset is a DataArray and can be accessed by name, such as `ds.red` or `ds.nir`.
Arithmetic operators including `+`, `-`, `*`, and `/` act element by element.
Before calculating, xarray aligns DataArrays by their dimension names and coordinate labels, preventing values from different dates or locations from being combined inadvertently.

The **Simple Ratio (SR)** compares near-infrared and red reflectance:

$$\text{SR} = \frac{\text{NIR}}{\text{Red}}$$

Healthy vegetation reflects near-infrared strongly and absorbs red light.
Its SR is therefore usually high (> 1), while bare ground tends to be closer to 1 and water tends to have a low value.

In [ ]:
sr = ds.nir / ds.red
sr

For every time and pixel coordinate, the calculation divides the near-infrared value by the corresponding red value.
The result, `sr`, is a new DataArray with the same `time`, `y`, and `x` dimensions and coordinates.

## G. Next steps

Notebook 05 applies these structures when drawing true-colour composites and single-band images: [`05_plotting.ipynb`](./05_plotting.ipynb).